# FactoryVisionAI model training

Runs pretrained YOLO baseline, PPE fine-tuning, and casting CNN / MobileNetV3 experiments. Logs and downloadable checkpoints are saved under `/kaggle/working/factoryvision_artifacts/`.


In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("ultralytics") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.3"],
        check=True,
    )


In [ ]:
import sys
__factoryvision_original_argv = sys.argv[:]
try:
    sys.argv = ['kaggle_train.py', '--epochs', '15', '--ppe-epochs', '30']
    exec(compile("import csv\nimport hashlib\nimport json\nimport logging\nimport platform\nimport random\nimport shutil\nimport sys\nimport time\nimport zipfile\nfrom contextlib import contextmanager, redirect_stderr, redirect_stdout\nfrom datetime import datetime\nfrom pathlib import Path\nfrom typing import Any, Generator, TextIO\n\nimport torch\nimport torchvision\nimport ultralytics\nimport yaml\nfrom PIL import Image\nfrom torch import nn\nfrom torch.utils.data import DataLoader, Dataset\nfrom torchvision import datasets, models, transforms\n\n\nQUALITY_CLASSES = (\"def_front\", \"ok_front\")\nPPE_CLASSES = (\n    \"Hardhat\",\n    \"Mask\",\n    \"NO-Hardhat\",\n    \"NO-Mask\",\n    \"NO-Safety Vest\",\n    \"Person\",\n    \"Safety Cone\",\n    \"Safety Vest\",\n    \"machinery\",\n    \"vehicle\",\n)\nOUTPUT_ROOT = Path(\"/kaggle/working/factoryvision_artifacts\")\nIMAGE_SIZE = 224\nLOGGER = logging.getLogger(\"factoryvision\")\n\n\nclass TeeStream:\n    def __init__(self, console: TextIO, log_file: TextIO) -> None:\n        self.console = console\n        self.log_file = log_file\n\n    def write(self, text: str) -> int:\n        self.console.write(text)\n        self.log_file.write(text)\n        return len(text)\n\n    def flush(self) -> None:\n        self.console.flush()\n        self.log_file.flush()\n\n    def isatty(self) -> bool:\n        return self.console.isatty()\n\n    def __getattr__(self, name: str) -> Any:\n        return getattr(self.console, name)\n\n\n@contextmanager\ndef capture_console_output(log_path: Path) -> Generator[None, None, None]:\n    with log_path.open(\"a\", encoding=\"utf-8\", buffering=1) as log_file:\n        with redirect_stdout(TeeStream(sys.stdout, log_file)):\n            with redirect_stderr(TeeStream(sys.stderr, log_file)):\n                yield\n\n\nclass CastingCNN(nn.Module):\n    def __init__(self, num_classes: int = len(QUALITY_CLASSES)) -> None:\n        super().__init__()\n        self.features = nn.Sequential(\n            nn.Conv2d(3, 32, kernel_size=3, padding=1),\n            nn.BatchNorm2d(32),\n            nn.ReLU(inplace=True),\n            nn.MaxPool2d(2),\n            nn.Conv2d(32, 64, kernel_size=3, padding=1),\n            nn.BatchNorm2d(64),\n            nn.ReLU(inplace=True),\n            nn.MaxPool2d(2),\n            nn.Conv2d(64, 128, kernel_size=3, padding=1),\n            nn.BatchNorm2d(128),\n            nn.ReLU(inplace=True),\n            nn.AdaptiveAvgPool2d((1, 1)),\n        )\n        self.classifier = nn.Sequential(\n            nn.Flatten(),\n            nn.Dropout(p=0.25),\n            nn.Linear(128, num_classes),\n        )\n\n    def forward(self, images: torch.Tensor) -> torch.Tensor:\n        return self.classifier(self.features(images))\n\n\nclass TransformedSubset(Dataset[tuple[torch.Tensor, int]]):\n    def __init__(\n        self,\n        dataset: datasets.ImageFolder,\n        indices: list[int],\n        transform: transforms.Compose,\n    ) -> None:\n        self.dataset = dataset\n        self.indices = indices\n        self.transform = transform\n\n    def __len__(self) -> int:\n        return len(self.indices)\n\n    def __getitem__(self, index: int) -> tuple[torch.Tensor, int]:\n        image_path, label = self.dataset.samples[self.indices[index]]\n        with Image.open(image_path) as image:\n            image_rgb = image.convert(\"RGB\")\n        return self.transform(image_rgb), label\n\n\ndef build_quality_model(architecture: str, pretrained: bool = False) -> nn.Module:\n    if architecture == \"cnn\":\n        return CastingCNN()\n    if architecture == \"mobilenet_v3_small\":\n        weights = models.MobileNet_V3_Small_Weights.DEFAULT if pretrained else None\n        model = models.mobilenet_v3_small(weights=weights)\n        input_features = model.classifier[-1].in_features\n        model.classifier[-1] = nn.Linear(input_features, len(QUALITY_CLASSES))\n        return model\n    raise ValueError(f\"Unknown quality architecture: {architecture}\")\n\n\ndef discover_ppe_root(input_root: Path) -> Path:\n    candidates = []\n    for train_path in input_root.rglob(\"train\"):\n        if not train_path.is_dir():\n            continue\n        candidate = train_path.parent\n        if all(\n            (candidate / split / kind).is_dir()\n            for split in (\"train\", \"valid\", \"test\")\n            for kind in (\"images\", \"labels\")\n        ):\n            candidates.append(candidate)\n    if not candidates:\n        raise FileNotFoundError(\n            f\"No PPE train/valid/test images and labels found under {input_root}.\"\n        )\n    return sorted(candidates, key=lambda path: len(path.parts))[0]\n\n\ndef discover_casting_root(input_root: Path) -> Path:\n    candidates = []\n    for train_path in input_root.rglob(\"train\"):\n        if not train_path.is_dir():\n            continue\n        candidate = train_path.parent\n        if all(\n            (candidate / split / class_name).is_dir()\n            for split in (\"train\", \"test\")\n            for class_name in QUALITY_CLASSES\n        ):\n            candidates.append(candidate)\n    if not candidates:\n        raise FileNotFoundError(\n            f\"No casting train/test class folders found under {input_root}.\"\n        )\n    return sorted(candidates, key=lambda path: len(path.parts))[0]\n\n\ndef validate_ppe_dataset(dataset_root: Path) -> None:\n    for split in (\"train\", \"valid\", \"test\"):\n        for kind in (\"images\", \"labels\"):\n            directory = dataset_root / split / kind\n            if not directory.is_dir() or not any(directory.iterdir()):\n                raise ValueError(f\"PPE dataset directory is missing or empty: {directory}\")\n\n\ndef validate_casting_dataset(dataset_root: Path) -> None:\n    for split in (\"train\", \"test\"):\n        for class_name in QUALITY_CLASSES:\n            directory = dataset_root / split / class_name\n            if not directory.is_dir() or not any(directory.iterdir()):\n                raise ValueError(\n                    f\"Casting dataset directory is missing or empty: {directory}\"\n                )\n\n\ndef setup_logging(output_root: Path) -> Path:\n    log_path = output_root / \"training.log\"\n    LOGGER.setLevel(logging.INFO)\n    LOGGER.handlers.clear()\n    formatter = logging.Formatter(\n        \"%(asctime)s %(levelname)s %(message)s\",\n        datefmt=\"%Y-%m-%d %H:%M:%S\",\n    )\n    file_handler = logging.FileHandler(log_path, encoding=\"utf-8\")\n    file_handler.setFormatter(formatter)\n    console_handler = logging.StreamHandler(sys.stdout)\n    console_handler.setFormatter(formatter)\n    LOGGER.addHandler(file_handler)\n    LOGGER.addHandler(console_handler)\n    return log_path\n\n\ndef inspect_ppe_annotations(dataset_root: Path) -> dict[str, Any]:\n    splits: dict[str, Any] = {}\n    for split in (\"train\", \"valid\", \"test\"):\n        image_dir = dataset_root / split / \"images\"\n        label_dir = dataset_root / split / \"labels\"\n        class_counts = {class_name: 0 for class_name in PPE_CLASSES}\n        image_stems = {\n            path.stem\n            for path in image_dir.iterdir()\n            if path.is_file()\n        }\n        label_stems = {path.stem for path in label_dir.glob(\"*.txt\")}\n        if image_stems != label_stems:\n            raise ValueError(\n                f\"PPE {split} images and label files do not match \"\n                f\"(images without labels: {len(image_stems - label_stems)}, \"\n                f\"labels without images: {len(label_stems - image_stems)}).\"\n            )\n        for label_path in label_dir.glob(\"*.txt\"):\n            for line_number, line in enumerate(\n                label_path.read_text(encoding=\"utf-8\").splitlines(),\n                start=1,\n            ):\n                fields = line.split()\n                if not fields:\n                    continue\n                if len(fields) != 5:\n                    raise ValueError(f\"Malformed YOLO annotation: {label_path}:{line_number}\")\n                class_id = int(fields[0])\n                coordinates = [float(value) for value in fields[1:]]\n                if not 0 <= class_id < len(PPE_CLASSES):\n                    raise ValueError(f\"Invalid PPE class id at {label_path}:{line_number}\")\n                if not all(0 <= value <= 1 for value in coordinates):\n                    raise ValueError(\n                        f\"Out-of-range YOLO coordinates at {label_path}:{line_number}\"\n                    )\n                class_counts[PPE_CLASSES[class_id]] += 1\n        splits[split] = {\n            \"image_count\": len(image_stems),\n            \"annotation_count\": sum(class_counts.values()),\n            \"class_counts\": class_counts,\n        }\n    return {\n        \"root\": str(dataset_root),\n        \"classes\": list(PPE_CLASSES),\n        \"splits\": splits,\n    }\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open(\"rb\") as file:\n        for chunk in iter(lambda: file.read(1024 * 1024), b\"\"):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef inspect_casting_dataset(dataset_root: Path) -> dict[str, Any]:\n    split_hashes: dict[str, dict[str, str]] = {}\n    split_counts: dict[str, dict[str, int]] = {}\n    for split in (\"train\", \"test\"):\n        split_hashes[split] = {}\n        split_counts[split] = {}\n        for class_name in QUALITY_CLASSES:\n            paths = [\n                path\n                for path in (dataset_root / split / class_name).rglob(\"*\")\n                if path.is_file()\n            ]\n            if not paths:\n                raise ValueError(f\"No images found for {split}/{class_name}.\")\n            split_counts[split][class_name] = len(paths)\n            for path in sorted(paths):\n                digest = sha256_file(path)\n                previous = split_hashes[split].get(digest)\n                if previous is not None:\n                    raise ValueError(\n                        f\"Duplicate casting image within the {split} split: \"\n                        f\"{previous} and {path}\"\n                    )\n                split_hashes[split][digest] = str(path)\n\n    leaked_hashes = set(split_hashes[\"train\"]) & set(split_hashes[\"test\"])\n    excluded_train_paths = []\n    for digest in sorted(leaked_hashes):\n        train_path = Path(split_hashes[\"train\"][digest])\n        test_path = Path(split_hashes[\"test\"][digest])\n        if train_path.parent.name != test_path.parent.name:\n            raise ValueError(\n                \"An exact duplicate casting image has conflicting train/test labels: \"\n                f\"{train_path} ({train_path.parent.name}) and \"\n                f\"{test_path} ({test_path.parent.name}).\"\n            )\n        excluded_train_paths.append(\n            train_path.relative_to(dataset_root).as_posix()\n        )\n    excluded_by_class = {\n        class_name: sum(\n            Path(path).parts[1] == class_name\n            for path in excluded_train_paths\n        )\n        for class_name in QUALITY_CLASSES\n    }\n    return {\n        \"root\": str(dataset_root),\n        \"classes\": list(QUALITY_CLASSES),\n        \"split_counts\": split_counts,\n        \"effective_train_counts\": {\n            class_name: split_counts[\"train\"][class_name] - excluded_by_class[class_name]\n            for class_name in QUALITY_CLASSES\n        },\n        \"exact_duplicates_across_train_test\": len(leaked_hashes),\n        \"train_paths_excluded_to_prevent_test_leakage\": excluded_train_paths,\n    }\n\n\ndef inspect_datasets(\n    ppe_root: Path,\n    casting_root: Path,\n    output_root: Path,\n) -> dict[str, Any]:\n    report = {\n        \"ppe\": inspect_ppe_annotations(ppe_root),\n        \"casting\": inspect_casting_dataset(casting_root),\n        \"casting_validation_split\": (\n            \"Stratified 20% holdout from train; original test split remains untouched.\"\n        ),\n    }\n    report_path = output_root / \"dataset_inspection.json\"\n    report_path.write_text(json.dumps(report, indent=2), encoding=\"utf-8\")\n    LOGGER.info(\"Dataset inspection report: %s\", report_path)\n    LOGGER.info(\"Dataset counts: %s\", json.dumps(report, sort_keys=True))\n    return report\n\n\ndef calculate_iou(\n    first: tuple[float, float, float, float],\n    second: tuple[float, float, float, float],\n) -> float:\n    intersection_width = max(min(first[2], second[2]) - max(first[0], second[0]), 0.0)\n    intersection_height = max(\n        min(first[3], second[3]) - max(first[1], second[1]),\n        0.0,\n    )\n    intersection = intersection_width * intersection_height\n    first_area = max(first[2] - first[0], 0.0) * max(first[3] - first[1], 0.0)\n    second_area = max(second[2] - second[0], 0.0) * max(\n        second[3] - second[1],\n        0.0,\n    )\n    union = first_area + second_area - intersection\n    return intersection / union if union else 0.0\n\n\ndef match_boxes(\n    predictions: list[tuple[float, float, float, float, float]],\n    ground_truth: list[tuple[float, float, float, float]],\n    iou_threshold: float,\n) -> tuple[int, int, int]:\n    matched: set[int] = set()\n    true_positive = 0\n    false_positive = 0\n    for prediction in sorted(predictions, key=lambda box: box[4], reverse=True):\n        overlaps = [\n            (calculate_iou(prediction[:4], truth), index)\n            for index, truth in enumerate(ground_truth)\n            if index not in matched\n        ]\n        best_iou, best_index = max(overlaps, default=(0.0, -1))\n        if best_iou >= iou_threshold:\n            true_positive += 1\n            matched.add(best_index)\n        else:\n            false_positive += 1\n    false_negative = len(ground_truth) - len(matched)\n    return true_positive, false_positive, false_negative\n\n\ndef read_person_annotations(label_path: Path, width: int, height: int) -> list[tuple[float, float, float, float]]:\n    boxes = []\n    person_class_id = PPE_CLASSES.index(\"Person\")\n    for line_number, line in enumerate(label_path.read_text(encoding=\"utf-8\").splitlines(), start=1):\n        fields = line.split()\n        if not fields:\n            continue\n        if len(fields) != 5:\n            raise ValueError(f\"Malformed YOLO annotation: {label_path}:{line_number}\")\n        if int(fields[0]) != person_class_id:\n            continue\n        center_x, center_y, box_width, box_height = (\n            float(value)\n            for value in fields[1:]\n        )\n        boxes.append(\n            (\n                (center_x - box_width / 2) * width,\n                (center_y - box_height / 2) * height,\n                (center_x + box_width / 2) * width,\n                (center_y + box_height / 2) * height,\n            )\n        )\n    return boxes\n\n\ndef evaluate_pretrained_person_baseline(\n    ppe_root: Path,\n    output_root: Path,\n    image_size: int,\n    max_visualizations: int = 12,\n) -> dict[str, Any]:\n    import cv2\n    from ultralytics import YOLO\n\n    test_image_dir = ppe_root / \"test\" / \"images\"\n    test_label_dir = ppe_root / \"test\" / \"labels\"\n    image_paths = sorted(\n        path\n        for path in test_image_dir.iterdir()\n        if path.is_file() and path.suffix.lower() in {\".jpg\", \".jpeg\", \".png\", \".bmp\"}\n    )\n    if not image_paths:\n        raise ValueError(f\"No PPE test images found in {test_image_dir}.\")\n\n    baseline_dir = output_root / \"baseline_pretrained_yolo\"\n    visualization_dir = baseline_dir / \"sample_predictions\"\n    visualization_dir.mkdir(parents=True, exist_ok=True)\n    model = YOLO(\"yolo11n.pt\")\n    class_names = model.names\n    class_items = class_names.items() if isinstance(class_names, dict) else enumerate(class_names)\n    person_id = next(\n        (int(index) for index, label in class_items if str(label).lower() == \"person\"),\n        None,\n    )\n    if person_id is None:\n        raise ValueError(\"Pretrained YOLO model does not expose a person class.\")\n\n    true_positive = false_positive = false_negative = 0\n    ground_truth_count = prediction_count = image_count = visualizations = 0\n    latency_total_ms = 0.0\n    for result in model.predict(\n        source=[str(path) for path in image_paths],\n        classes=[person_id],\n        imgsz=image_size,\n        conf=0.25,\n        stream=True,\n        verbose=False,\n    ):\n        image_path = Path(str(result.path))\n        image = result.orig_img\n        height, width = image.shape[:2]\n        annotation_path = test_label_dir / f\"{image_path.stem}.txt\"\n        ground_truth = (\n            read_person_annotations(annotation_path, width, height)\n            if annotation_path.is_file()\n            else []\n        )\n        predictions = []\n        if result.boxes is not None:\n            predictions = [\n                (\n                    float(box[0]),\n                    float(box[1]),\n                    float(box[2]),\n                    float(box[3]),\n                    float(confidence),\n                )\n                for box, confidence in zip(\n                    result.boxes.xyxy.cpu().tolist(),\n                    result.boxes.conf.cpu().tolist(),\n                )\n            ]\n        tp, fp, fn = match_boxes(predictions, ground_truth, iou_threshold=0.5)\n        true_positive += tp\n        false_positive += fp\n        false_negative += fn\n        ground_truth_count += len(ground_truth)\n        prediction_count += len(predictions)\n        image_count += 1\n        latency_total_ms += float(result.speed.get(\"inference\", 0.0))\n        if visualizations < max_visualizations:\n            output_path = visualization_dir / image_path.name\n            if not cv2.imwrite(str(output_path), result.plot()):\n                raise OSError(f\"Could not save baseline prediction to {output_path}.\")\n            visualizations += 1\n\n    precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else 0.0\n    recall = true_positive / (true_positive + false_negative) if true_positive + false_negative else 0.0\n    metrics = {\n        \"model\": \"YOLO11n pretrained COCO\",\n        \"task\": \"person-only baseline (not PPE compliance)\",\n        \"split\": \"test\",\n        \"images\": image_count,\n        \"ground_truth_persons\": ground_truth_count,\n        \"predicted_persons\": prediction_count,\n        \"true_positive\": true_positive,\n        \"false_positive\": false_positive,\n        \"false_negative\": false_negative,\n        \"precision\": precision,\n        \"recall\": recall,\n        \"f1\": 2 * precision * recall / (precision + recall) if precision + recall else 0.0,\n        \"mean_inference_latency_ms\": latency_total_ms / image_count if image_count else 0.0,\n        \"sample_visualizations\": visualizations,\n    }\n    (baseline_dir / \"metrics.json\").write_text(json.dumps(metrics, indent=2), encoding=\"utf-8\")\n    LOGGER.info(\"Pretrained YOLO baseline: %s\", json.dumps(metrics, sort_keys=True))\n    return metrics\n\n\ndef build_ppe_config(ppe_root: Path, output_root: Path) -> Path:\n    config = {\n        \"path\": str(ppe_root.resolve()),\n        \"train\": \"train/images\",\n        \"val\": \"valid/images\",\n        \"test\": \"test/images\",\n        \"nc\": len(PPE_CLASSES),\n        \"names\": list(PPE_CLASSES),\n    }\n    config_path = output_root / \"ppe_data.yaml\"\n    config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding=\"utf-8\")\n    return config_path\n\n\ndef train_ppe_yolo(\n    ppe_root: Path,\n    output_root: Path,\n    epochs: int,\n    image_size: int,\n    batch_size: int,\n) -> tuple[Path, dict[str, float]]:\n    from ultralytics import YOLO\n\n    config_path = build_ppe_config(ppe_root, output_root)\n    run_name = f\"ppe_yolo_{datetime.now().strftime('%Y%m%d_%H%M%S')}\"\n    run_root = output_root / \"yolo_runs\"\n    device = 0 if torch.cuda.is_available() else \"cpu\"\n    LOGGER.info(\n        \"Fine-tuning PPE YOLO: epochs=%d image_size=%d batch_size=%d device=%s\",\n        epochs,\n        image_size,\n        batch_size,\n        device,\n    )\n    model = YOLO(\"yolo11n.pt\")\n    model.train(\n        data=str(config_path),\n        epochs=epochs,\n        imgsz=image_size,\n        batch=batch_size,\n        device=device,\n        workers=2,\n        project=str(run_root),\n        name=run_name,\n        patience=10,\n        save=True,\n    )\n    best_weights = run_root / run_name / \"weights\" / \"best.pt\"\n    if not best_weights.is_file():\n        raise FileNotFoundError(f\"Best PPE weights were not produced: {best_weights}\")\n\n    checkpoint_dir = output_root / \"checkpoints\"\n    checkpoint_dir.mkdir(parents=True, exist_ok=True)\n    destination = checkpoint_dir / \"ppe_yolo.pt\"\n    shutil.copy2(best_weights, destination)\n    test_results = YOLO(str(destination)).val(\n        data=str(config_path),\n        split=\"test\",\n        imgsz=image_size,\n        batch=batch_size,\n        device=device,\n        project=str(run_root),\n        name=f\"{run_name}_test\",\n    )\n    metrics = {\n        str(key): float(value)\n        for key, value in test_results.results_dict.items()\n    }\n    (output_root / \"ppe_test_metrics.json\").write_text(\n        json.dumps(metrics, indent=2),\n        encoding=\"utf-8\",\n    )\n    LOGGER.info(\"Fine-tuned PPE test metrics: %s\", json.dumps(metrics, sort_keys=True))\n    return destination, metrics\n\n\ndef make_transforms(training: bool) -> transforms.Compose:\n    steps: list[Any] = [transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))]\n    if training:\n        steps.append(transforms.RandomHorizontalFlip())\n    steps.extend(\n        [\n            transforms.ToTensor(),\n            transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),\n        ]\n    )\n    return transforms.Compose(steps)\n\n\ndef make_quality_loaders(\n    casting_root: Path,\n    batch_size: int,\n    seed: int,\n    excluded_train_paths: set[str],\n) -> tuple[DataLoader, DataLoader, DataLoader]:\n    train_dataset = datasets.ImageFolder(casting_root / \"train\")\n    test_dataset = datasets.ImageFolder(\n        casting_root / \"test\",\n        transform=make_transforms(training=False),\n    )\n    if tuple(train_dataset.classes) != QUALITY_CLASSES:\n        raise ValueError(\n            f\"Expected casting classes {QUALITY_CLASSES}, got {train_dataset.classes}.\"\n        )\n    if tuple(test_dataset.classes) != QUALITY_CLASSES:\n        raise ValueError(\n            f\"Train/test casting class order differs: {train_dataset.classes}, \"\n            f\"{test_dataset.classes}.\"\n        )\n    excluded_samples = [\n        sample\n        for sample in train_dataset.samples\n        if Path(sample[0]).relative_to(casting_root).as_posix()\n        in excluded_train_paths\n    ]\n    if len(excluded_samples) != len(excluded_train_paths):\n        raise ValueError(\n            \"Casting inspection reported duplicate training paths that do not match \"\n            \"the loaded ImageFolder samples.\"\n        )\n    retained_train_samples = [\n        sample\n        for sample in train_dataset.samples\n        if Path(sample[0]).relative_to(casting_root).as_posix()\n        not in excluded_train_paths\n    ]\n    train_dataset.samples = retained_train_samples\n    train_dataset.imgs = retained_train_samples\n    train_dataset.targets = [label for _, label in retained_train_samples]\n\n    class_indices = {\n        class_id: [\n            index\n            for index, (_, label) in enumerate(train_dataset.samples)\n            if label == class_id\n        ]\n        for class_id in range(len(QUALITY_CLASSES))\n    }\n    train_indices: list[int] = []\n    validation_indices: list[int] = []\n    for class_id, indices in class_indices.items():\n        random.Random(seed + class_id).shuffle(indices)\n        validation_count = max(1, round(len(indices) * 0.2))\n        validation_indices.extend(indices[:validation_count])\n        train_indices.extend(indices[validation_count:])\n\n    train_data = TransformedSubset(\n        train_dataset,\n        train_indices,\n        make_transforms(training=True),\n    )\n    validation_data = TransformedSubset(\n        train_dataset,\n        validation_indices,\n        make_transforms(training=False),\n    )\n    worker_count = min(4, torch.get_num_threads())\n    return (\n        DataLoader(\n            train_data,\n            batch_size=batch_size,\n            shuffle=True,\n            num_workers=worker_count,\n            pin_memory=torch.cuda.is_available(),\n        ),\n        DataLoader(\n            validation_data,\n            batch_size=batch_size,\n            shuffle=False,\n            num_workers=worker_count,\n            pin_memory=torch.cuda.is_available(),\n        ),\n        DataLoader(\n            test_dataset,\n            batch_size=batch_size,\n            shuffle=False,\n            num_workers=worker_count,\n            pin_memory=torch.cuda.is_available(),\n        ),\n    )\n\n\ndef calculate_quality_metrics(\n    model: nn.Module,\n    loader: DataLoader,\n    device: torch.device,\n) -> dict[str, Any]:\n    confusion = torch.zeros(\n        len(QUALITY_CLASSES),\n        len(QUALITY_CLASSES),\n        dtype=torch.int64,\n    )\n    inference_seconds = 0.0\n    image_count = 0\n    model.eval()\n    with torch.inference_mode():\n        for images, labels in loader:\n            images = images.to(device, non_blocking=True)\n            if device.type == \"cuda\":\n                torch.cuda.synchronize(device)\n            started_at = time.perf_counter()\n            predictions = model(images).argmax(dim=1).cpu()\n            if device.type == \"cuda\":\n                torch.cuda.synchronize(device)\n            inference_seconds += time.perf_counter() - started_at\n            labels = labels.cpu()\n            image_count += int(labels.size(0))\n            for actual, predicted in zip(labels, predictions):\n                confusion[int(actual), int(predicted)] += 1\n\n    precision_by_class: dict[str, float] = {}\n    recall_by_class: dict[str, float] = {}\n    f1_by_class: dict[str, float] = {}\n    for index, class_name in enumerate(QUALITY_CLASSES):\n        true_positive = int(confusion[index, index])\n        false_positive = int(confusion[:, index].sum()) - true_positive\n        false_negative = int(confusion[index, :].sum()) - true_positive\n        precision_by_class[class_name] = (\n            true_positive / (true_positive + false_positive)\n            if true_positive + false_positive\n            else 0.0\n        )\n        recall_by_class[class_name] = (\n            true_positive / (true_positive + false_negative)\n            if true_positive + false_negative\n            else 0.0\n        )\n        f1_denominator = 2 * true_positive + false_positive + false_negative\n        f1_by_class[class_name] = (\n            2 * true_positive / f1_denominator if f1_denominator else 0.0\n        )\n\n    total = int(confusion.sum())\n    return {\n        \"accuracy\": int(confusion.diag().sum()) / total if total else 0.0,\n        \"precision_macro\": sum(precision_by_class.values()) / len(QUALITY_CLASSES),\n        \"recall_macro\": sum(recall_by_class.values()) / len(QUALITY_CLASSES),\n        \"macro_f1\": sum(f1_by_class.values()) / len(QUALITY_CLASSES),\n        \"per_class_precision\": precision_by_class,\n        \"per_class_recall\": recall_by_class,\n        \"per_class_f1\": f1_by_class,\n        \"confusion_matrix\": confusion.tolist(),\n        \"classes\": list(QUALITY_CLASSES),\n        \"mean_inference_latency_ms\": (\n            inference_seconds * 1000 / image_count if image_count else 0.0\n        ),\n    }\n\n\ndef train_quality_model(\n    architecture: str,\n    casting_root: Path,\n    output_root: Path,\n    epochs: int,\n    batch_size: int,\n    learning_rate: float,\n    seed: int,\n    excluded_train_paths: set[str],\n) -> tuple[Path, dict[str, Any]]:\n    random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    train_loader, validation_loader, test_loader = make_quality_loaders(\n        casting_root,\n        batch_size,\n        seed,\n        excluded_train_paths,\n    )\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    model = build_quality_model(\n        architecture,\n        pretrained=architecture == \"mobilenet_v3_small\",\n    ).to(device)\n    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)\n    loss_function = nn.CrossEntropyLoss()\n    checkpoint_dir = output_root / \"checkpoints\"\n    checkpoint_dir.mkdir(parents=True, exist_ok=True)\n    checkpoint_path = checkpoint_dir / f\"casting_{architecture}.pth\"\n    history_path = output_root / f\"{architecture}_history.csv\"\n    history_fields = [\n        \"epoch\",\n        \"train_loss\",\n        \"validation_accuracy\",\n        \"validation_precision_macro\",\n        \"validation_recall_macro\",\n        \"validation_macro_f1\",\n        \"validation_def_front_f1\",\n        \"validation_ok_front_f1\",\n    ]\n    best_validation_f1 = -1.0\n    LOGGER.info(\n        \"Training quality model %s for %d epochs on %s\",\n        architecture,\n        epochs,\n        device,\n    )\n    with history_path.open(\"w\", newline=\"\", encoding=\"utf-8\") as file:\n        writer = csv.DictWriter(file, fieldnames=history_fields)\n        writer.writeheader()\n        for epoch in range(1, epochs + 1):\n            model.train()\n            total_loss = 0.0\n            for images, labels in train_loader:\n                images = images.to(device, non_blocking=True)\n                labels = labels.to(device, non_blocking=True)\n                optimizer.zero_grad(set_to_none=True)\n                loss = loss_function(model(images), labels)\n                loss.backward()\n                optimizer.step()\n                total_loss += float(loss.item())\n\n            validation_metrics = calculate_quality_metrics(\n                model,\n                validation_loader,\n                device,\n            )\n            average_loss = total_loss / max(len(train_loader), 1)\n            writer.writerow(\n                {\n                    \"epoch\": epoch,\n                    \"train_loss\": average_loss,\n                    \"validation_accuracy\": validation_metrics[\"accuracy\"],\n                    \"validation_precision_macro\": validation_metrics[\"precision_macro\"],\n                    \"validation_recall_macro\": validation_metrics[\"recall_macro\"],\n                    \"validation_macro_f1\": validation_metrics[\"macro_f1\"],\n                    \"validation_def_front_f1\": validation_metrics[\"per_class_f1\"][\"def_front\"],\n                    \"validation_ok_front_f1\": validation_metrics[\"per_class_f1\"][\"ok_front\"],\n                }\n            )\n            file.flush()\n            LOGGER.info(\n                \"%s epoch %d/%d loss=%.4f validation_accuracy=%.4f \"\n                \"validation_macro_f1=%.4f\",\n                architecture,\n                epoch,\n                epochs,\n                average_loss,\n                validation_metrics[\"accuracy\"],\n                validation_metrics[\"macro_f1\"],\n            )\n            if validation_metrics[\"macro_f1\"] > best_validation_f1:\n                best_validation_f1 = validation_metrics[\"macro_f1\"]\n                torch.save(\n                    {\n                        \"architecture\": architecture,\n                        \"classes\": list(QUALITY_CLASSES),\n                        \"state_dict\": {\n                            name: tensor.detach().cpu()\n                            for name, tensor in model.state_dict().items()\n                        },\n                        \"image_size\": IMAGE_SIZE,\n                        \"validation_macro_f1\": best_validation_f1,\n                        \"seed\": seed,\n                    },\n                    checkpoint_path,\n                )\n\n    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)\n    model.load_state_dict(checkpoint[\"state_dict\"])\n    validation_metrics = calculate_quality_metrics(model, validation_loader, device)\n    test_metrics = calculate_quality_metrics(model, test_loader, device)\n    results = {\n        \"checkpoint\": checkpoint_path.name,\n        \"validation\": validation_metrics,\n        \"test\": test_metrics,\n    }\n    metrics_path = output_root / f\"{architecture}_metrics.json\"\n    metrics_path.write_text(json.dumps(results, indent=2), encoding=\"utf-8\")\n    LOGGER.info(\"%s results: %s\", architecture, json.dumps(results, sort_keys=True))\n    return checkpoint_path, results\n\n\ndef package_artifacts(\n    output_root: Path,\n    ppe_metrics: dict[str, float],\n    baseline_metrics: dict[str, Any],\n    quality_results: dict[str, dict[str, Any]],\n    best_architecture: str,\n) -> Path:\n    best_quality_checkpoint = output_root / \"checkpoints\" / \"casting_best.pth\"\n    shutil.copy2(\n        output_root / \"checkpoints\" / f\"casting_{best_architecture}.pth\",\n        best_quality_checkpoint,\n    )\n    report = {\n        \"pretrained_person_baseline\": baseline_metrics,\n        \"ppe_test\": ppe_metrics,\n        \"quality_models\": quality_results,\n        \"best_quality_architecture\": best_architecture,\n        \"selection_metric\": \"validation macro-F1\",\n        \"test_split_used_for_selection\": False,\n    }\n    report_path = output_root / \"training_metrics.json\"\n    report_path.write_text(json.dumps(report, indent=2), encoding=\"utf-8\")\n\n    included_files: dict[str, Path] = {}\n    for folder_name in (\"checkpoints\", \"baseline_pretrained_yolo\", \"yolo_runs\"):\n        folder = output_root / folder_name\n        if folder.is_dir():\n            for file_path in folder.rglob(\"*\"):\n                if file_path.is_file():\n                    included_files[file_path.relative_to(output_root).as_posix()] = file_path\n    for path in output_root.iterdir():\n        if path.is_file() and path.name != \"artifact_manifest.json\":\n            included_files[path.relative_to(output_root).as_posix()] = path\n    included_files[\"training_metrics.json\"] = report_path\n\n    required = {\n        \"checkpoints/ppe_yolo.pt\",\n        \"checkpoints/casting_best.pth\",\n        \"training.log\",\n        \"console.log\",\n        \"dataset_inspection.json\",\n        \"baseline_pretrained_yolo/metrics.json\",\n    }\n    if missing := required - included_files.keys():\n        raise FileNotFoundError(\n            f\"Training output bundle is incomplete: {sorted(missing)}\"\n        )\n\n    manifest = {\n        \"format_version\": 1,\n        \"quality_architecture\": best_architecture,\n        \"files\": {\n            name: sha256_file(path)\n            for name, path in included_files.items()\n        },\n    }\n    manifest_path = output_root / \"artifact_manifest.json\"\n    manifest_path.write_text(json.dumps(manifest, indent=2), encoding=\"utf-8\")\n    included_files[\"artifact_manifest.json\"] = manifest_path\n    archive_path = output_root / \"factoryvision_kaggle_models.zip\"\n    with zipfile.ZipFile(\n        archive_path,\n        mode=\"w\",\n        compression=zipfile.ZIP_DEFLATED,\n    ) as archive:\n        for name, path in included_files.items():\n            archive.write(path, arcname=name)\n    return archive_path\n\n\ndef run_training() -> Path:\n    import argparse\n    from ultralytics import YOLO\n\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--input-root\", type=Path, default=Path(\"/kaggle/input\"))\n    parser.add_argument(\"--output-root\", type=Path, default=OUTPUT_ROOT)\n    parser.add_argument(\"--epochs\", type=int, default=15)\n    parser.add_argument(\"--ppe-epochs\", type=int, default=30)\n    parser.add_argument(\"--image-size\", type=int, default=640)\n    parser.add_argument(\"--ppe-batch-size\", type=int, default=16)\n    parser.add_argument(\"--quality-batch-size\", type=int, default=64)\n    parser.add_argument(\"--learning-rate\", type=float, default=0.001)\n    parser.add_argument(\"--seed\", type=int, default=42)\n    args = parser.parse_args()\n    if min(args.epochs, args.ppe_epochs, args.image_size, args.ppe_batch_size, args.quality_batch_size) < 1:\n        parser.error(\"Epoch counts, image size, and batch sizes must be positive.\")\n    if args.learning_rate <= 0:\n        parser.error(\"Learning rate must be greater than zero.\")\n\n    output_root = args.output_root.resolve()\n    output_root.mkdir(parents=True, exist_ok=True)\n    log_path = output_root / \"training.log\"\n    runtime_metadata = {\n        \"python\": sys.version,\n        \"platform\": platform.platform(),\n        \"torch\": torch.__version__,\n        \"torchvision\": torchvision.__version__,\n        \"ultralytics\": ultralytics.__version__,\n        \"pillow\": Image.__version__,\n        \"pyyaml\": yaml.__version__,\n        \"cuda_available\": torch.cuda.is_available(),\n        \"cuda_version\": torch.version.cuda,\n        \"cuda_device\": (\n            torch.cuda.get_device_name(0) if torch.cuda.is_available() else None\n        ),\n        \"training_arguments\": {\n            \"epochs\": args.epochs,\n            \"ppe_epochs\": args.ppe_epochs,\n            \"image_size\": args.image_size,\n            \"ppe_batch_size\": args.ppe_batch_size,\n            \"quality_batch_size\": args.quality_batch_size,\n            \"learning_rate\": args.learning_rate,\n            \"seed\": args.seed,\n        },\n        \"reproducibility\": {\n            \"ppe_seed\": 0,\n            \"ppe_deterministic\": True,\n            \"quality_seed\": args.seed,\n            \"quality_validation_fraction\": 0.2,\n            \"quality_selection_metric\": \"validation macro-F1\",\n            \"test_split_used_for_model_selection\": False,\n        },\n    }\n    (output_root / \"runtime_metadata.json\").write_text(\n        json.dumps(runtime_metadata, indent=2),\n        encoding=\"utf-8\",\n    )\n    LOGGER.setLevel(logging.INFO)\n    LOGGER.handlers.clear()\n    formatter = logging.Formatter(\"%(asctime)s %(levelname)s %(message)s\")\n    file_handler = logging.FileHandler(log_path, encoding=\"utf-8\")\n    file_handler.setFormatter(formatter)\n    console_handler = logging.StreamHandler(sys.stdout)\n    console_handler.setFormatter(formatter)\n    LOGGER.addHandler(file_handler)\n    LOGGER.addHandler(console_handler)\n\n    with capture_console_output(output_root / \"console.log\"):\n        try:\n            ppe_root = discover_ppe_root(args.input_root)\n            casting_root = discover_casting_root(args.input_root)\n            validate_ppe_dataset(ppe_root)\n            validate_casting_dataset(casting_root)\n            inspection = inspect_datasets(ppe_root, casting_root, output_root)\n\n            baseline_metrics = evaluate_pretrained_person_baseline(\n                ppe_root,\n                output_root,\n                args.image_size,\n            )\n            config_path = build_ppe_config(ppe_root, output_root)\n            run_name = f\"ppe_yolo_{datetime.now().strftime('%Y%m%d_%H%M%S')}\"\n            run_root = output_root / \"yolo_runs\"\n            device = 0 if torch.cuda.is_available() else \"cpu\"\n            LOGGER.info(\n                \"Starting PPE YOLO fine-tuning for %d epochs on %s\",\n                args.ppe_epochs,\n                device,\n            )\n            detector = YOLO(\"yolo11n.pt\")\n            detector.train(\n                data=str(config_path),\n                epochs=args.ppe_epochs,\n                imgsz=args.image_size,\n                batch=args.ppe_batch_size,\n                device=device,\n                workers=2,\n                project=str(run_root),\n                name=run_name,\n                patience=10,\n                save=True,\n            )\n            best_ppe = run_root / run_name / \"weights\" / \"best.pt\"\n            if not best_ppe.is_file():\n                raise FileNotFoundError(f\"Best PPE checkpoint missing: {best_ppe}\")\n            checkpoint_dir = output_root / \"checkpoints\"\n            checkpoint_dir.mkdir(parents=True, exist_ok=True)\n            ppe_checkpoint = checkpoint_dir / \"ppe_yolo.pt\"\n            shutil.copy2(best_ppe, ppe_checkpoint)\n            ppe_test = YOLO(str(ppe_checkpoint)).val(\n                data=str(config_path),\n                split=\"test\",\n                imgsz=args.image_size,\n                batch=args.ppe_batch_size,\n                device=device,\n                project=str(run_root),\n                name=f\"{run_name}_test\",\n            )\n            ppe_metrics = {\n                str(key): float(value)\n                for key, value in ppe_test.results_dict.items()\n            }\n            (output_root / \"ppe_test_metrics.json\").write_text(\n                json.dumps(ppe_metrics, indent=2),\n                encoding=\"utf-8\",\n            )\n            LOGGER.info(\"PPE test results: %s\", json.dumps(ppe_metrics, sort_keys=True))\n\n            quality_results: dict[str, dict[str, Any]] = {}\n            for architecture in (\"cnn\", \"mobilenet_v3_small\"):\n                _, results = train_quality_model(\n                    architecture,\n                    casting_root,\n                    output_root,\n                    epochs=args.epochs,\n                    batch_size=args.quality_batch_size,\n                    learning_rate=args.learning_rate,\n                    seed=args.seed,\n                    excluded_train_paths=set(\n                        inspection[\"casting\"][\n                            \"train_paths_excluded_to_prevent_test_leakage\"\n                        ]\n                    ),\n                )\n                quality_results[architecture] = results\n            best_architecture = max(\n                quality_results,\n                key=lambda architecture: quality_results[architecture][\"validation\"][\"macro_f1\"],\n            )\n            archive_path = package_artifacts(\n                output_root,\n                ppe_metrics,\n                baseline_metrics,\n                quality_results,\n                best_architecture,\n            )\n            LOGGER.info(\"Input inspection: %s\", json.dumps(inspection, sort_keys=True))\n            LOGGER.info(\"Best quality architecture: %s\", best_architecture)\n            LOGGER.info(\"Training outputs: %s\", output_root)\n            LOGGER.info(\"Model bundle: %s\", archive_path)\n            return archive_path\n        except Exception:\n            LOGGER.exception(\"FactoryVisionAI Kaggle training failed.\")\n            raise\n\n\ndef main() -> None:\n    run_training()\n\n\nif __name__ == \"__main__\":\n    main()\n", 'kaggle_train.py', 'exec'), {'__name__': '__main__'})
finally:
    sys.argv = __factoryvision_original_argv
